In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as dt

In [2]:
# Chemin absolu (ou relatif) vers le dossier de travail
CHEMIN = "./"

In [3]:
data = pd.read_csv(CHEMIN + "Dati_sensori_aria_dal_2018_20251106.csv")
data.head()

,idSensore,Data,Valore,Stato,idOperatore
0,5504,01/01/2018 01:00:00,56.9,VA,1
1,5504,01/01/2018 02:00:00,53.8,VA,1
2,5504,01/01/2018 03:00:00,68,VA,1
3,5504,01/01/2018 04:00:00,60.1,VA,1
4,5504,01/01/2018 05:00:00,59.7,VA,1


In [4]:
data.shape

(20201649, 5)

## Filter Ozone Sensors

In [6]:
sensors = pd.read_csv(CHEMIN + "Stazioni_qualità_dell’aria_20251106.csv")
sensors.head()

,IdSensore,NomeTipoSensore,UnitaMisura,Idstazione,NomeStazione,Quota,Provincia,Comune,Storico,DataStart,DataStop,Utm_Nord,UTM_Est,lat,lng,Location
0,5710,Ozono,µg/m³,544,Cormano v. Edison,153.0,MI,Cormano,N,03/11/1994,NaN,5044180,512693,45.547597,9.166983,POINT (9.166983 45.547597)
1,6901,Ozono,µg/m³,668,Mantova Lunetta2,25.0,MN,Mantova,S,01/01/2003,01/01/2018,5002119,643358,45.157994,10.823940,POINT (10.82394014 45.15799358)
2,10320,PM10 (SM2005),µg/m³,548,Milano v.Senato,118.0,MI,Milano,N,10/08/2007,NaN,5035238,515435,45.470501,9.197461,POINT (9.19746075 45.470501)
3,10013,Particelle sospese PM2.5,µg/m³,576,Merate v. Madonna di Loreto,279.0,LC,Merate,N,10/09/2006,NaN,5060569,531623,45.697960,9.406192,POINT (9.40619194 45.69795956)
4,5664,Biossido di Zolfo,µg/m³,612,Filago v.Fermi,176.0,BG,Filago,S,17/07/1992,NaN,5052474,543382,45.624465,9.556516,POINT (9.55651564 45.62446461)


In [7]:
ozone_sensors = sensors[ sensors["NomeTipoSensore"] == "Ozono"]

93 sensors for Ozone

In [9]:
ozone_sensors_id = ozone_sensors["IdSensore"].unique()  # PS le unique ne sert à rien car les id sont tous différents

In [10]:
data_ozone = data[data["idSensore"].isin(ozone_sensors_id)]

In [11]:
list_stations = data_ozone["idSensore"].unique()
n_stations = len(list_stations)
n_stations

51

Only 51 stations cause the others stopped before 2018-01-01, so they were not in the data file

In [13]:
data_ozone.head()

,idSensore,Data,Valore,Stato,idOperatore
3292907,5707,01/01/2018 01:00:00,0.5,VA,1
3292908,5707,01/01/2018 02:00:00,0.6,VA,1
3292909,5707,01/01/2018 03:00:00,0.8,VA,1
3292910,5707,01/01/2018 04:00:00,6.9,VA,1
3292911,5707,01/01/2018 05:00:00,15.1,VA,1


In [14]:
data_ozone.shape

(3034092, 5)

## Compute Daily maximum

In [18]:
date = pd.to_datetime(data_ozone["Data"], format="%d/%m/%Y %H:%M:%S")

In [19]:
data_ozone["Data"] = date

C:\Users\alexa\AppData\Local\Temp\ipykernel_57864\2423058577.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_ozone["Data"] = date


In [20]:
data_ozone.head()

,idSensore,Data,Valore,Stato,idOperatore
3292907,5707,2018-01-01 01:00:00,0.5,VA,1
3292908,5707,2018-01-01 02:00:00,0.6,VA,1
3292909,5707,2018-01-01 03:00:00,0.8,VA,1
3292910,5707,2018-01-01 04:00:00,6.9,VA,1
3292911,5707,2018-01-01 05:00:00,15.1,VA,1


In [21]:
data_ozone = data_ozone[data_ozone["Stato"] == "VA"]
data_ozone["Valore"] = pd.to_numeric(data_ozone["Valore"], errors="coerce")


In [35]:
data_ozone["week"] = data_ozone["Data"].dt.to_period("W")

In [37]:
weekly_max = (
    data_ozone
    .groupby(["idSensore", "week"])["Valore"]
    .max()
    .reset_index()
)

weekly_max.head()


,idSensore,week,Valore
0,5707,2018-01-01/2018-01-07,78.4
1,5707,2018-01-08/2018-01-14,59.6
2,5707,2018-01-15/2018-01-21,79.7
3,5707,2018-01-22/2018-01-28,49.6
4,5707,2018-01-29/2018-02-04,55.7


In [39]:
weekly_max.shape

(18231, 3)

## Add Covariates

In [42]:
stations_location = sensors[sensors["IdSensore"].isin(list_stations)][["IdSensore", "lat", "lng"]]
stations_infos = weekly_max.groupby(["idSensore"]).week.agg(["min", "max"]).reset_index().merge(stations_location, left_on='idSensore', right_on='IdSensore')
stations_infos

,idSensore,min,max,IdSensore,lat,lng
0,5707,2018-01-01/2018-01-07,2024-12-30/2025-01-05,5707,45.548521,8.847327
1,5710,2018-01-01/2018-01-07,2024-12-30/2025-01-05,5710,45.547597,9.166983
2,5717,2018-01-01/2018-01-07,2024-12-30/2025-01-05,5717,45.483632,9.327362
3,5718,2018-01-01/2018-01-07,2023-10-02/2023-10-08,5718,45.462421,8.880214
4,5719,2018-01-01/2018-01-07,2023-10-02/2023-10-08,5719,45.660992,9.160225
5,5721,2018-01-01/2018-01-07,2024-12-30/2025-01-05,5721,45.281964,8.988576
6,5725,2018-01-01/2018-01-07,2024-12-30/2025-01-05,5725,45.463349,9.195325
7,5730,2018-01-01/2018-01-07,2024-07-08/2024-07-14,5730,45.808574,9.221779
8,5732,2018-01-01/2018-01-07,2024-12-30/2025-01-05,5732,46.138141,9.384687
9,5735,2018-01-01/2018-01-07,2022-03-14/2022-03-20,5735,45.697960,9.406192


In [124]:
stations_infos['elevation'] = ''
stations_infos['timezone'] = ''
stations_infos[['lat', 'lng', 'elevation', 'timezone', 'min', 'max']].to_csv(CHEMIN + "stations_infos.csv", header=False, index=False)

In [125]:
open_meteo_results = pd.read_csv(CHEMIN + "open_meteo_results.csv", 
                                 names=["idSensore", "day", "maxTemp", "minTemp", "rainSum", "windSpeed"], 
                                 skiprows=1)
date = pd.to_datetime(open_meteo_results["day"], format="%Y-%m-%d")
open_meteo_results['day'] = date
open_meteo_results["day"] = open_meteo_results["day"].dt.date
open_meteo_results.head()

,idSensore,day,maxTemp,minTemp,rainSum,windSpeed
0,0,2018-01-01,6.4,-1.0,3.3,11.9
1,0,2018-01-02,12.0,-0.6,0.0,15.5
2,0,2018-01-03,7.8,-1.3,1.7,21.3
3,0,2018-01-04,11.2,1.3,3.1,13.0
4,0,2018-01-05,6.9,-0.5,0.4,9.4


In [126]:
open_meteo_results["idSensore"] = open_meteo_results["idSensore"].map(stations_infos["idSensore"])
open_meteo_results.head()

,idSensore,day,maxTemp,minTemp,rainSum,windSpeed
0,5707,2018-01-01,6.4,-1.0,3.3,11.9
1,5707,2018-01-02,12.0,-0.6,0.0,15.5
2,5707,2018-01-03,7.8,-1.3,1.7,21.3
3,5707,2018-01-04,11.2,1.3,3.1,13.0
4,5707,2018-01-05,6.9,-0.5,0.4,9.4


In [127]:
final_data = daily_max.merge(open_meteo_results, how='inner', on=['idSensore','day'])

final_data.rename(columns={"idSensore": "idSensor", "Valore": "maxOzone"}, inplace=True)
final_data = final_data[["idSensor", "day", "maxTemp", "minTemp", "rainSum", "windSpeed", "maxOzone"]]

print(f"shape: {final_data.shape}")
final_data.head()

shape: (125782, 7)


,idSensor,day,maxTemp,minTemp,rainSum,windSpeed,maxOzone
0,5707,2018-01-01,6.4,-1.0,3.3,11.9,15.8
1,5707,2018-01-02,12.0,-0.6,0.0,15.5,71.2
2,5707,2018-01-03,7.8,-1.3,1.7,21.3,68.6
3,5707,2018-01-04,11.2,1.3,3.1,13.0,78.4
4,5707,2018-01-05,6.9,-0.5,0.4,9.4,3.9


In [128]:
final_data.to_csv(CHEMIN + "daily_ozone_and_covariates.csv", header=True, index=False)